In [3]:
from re import search

from dotenv import load_dotenv
load_dotenv()

True

# 1.0 自定义工具
## 1.1 基于tool描述工具

In [11]:
from langchain.tools import tool


@tool("square_root",description="calculate square root")
def square_root(x:float)->float:
    return x**0.5

## 1.2 使用函数名和文档注释描述工具

In [12]:
@tool
def square_root(x:float)->float:
    """calculate square root"""
    return x**0.5

In [13]:
@tool
def get_weather(location:str,units:str="celsius", include_forecast:bool=False)->str:
    """
    Get weather data and optionally include forecast
    args:
    location: location of weather data
    units: unit of degrees
    include_forecast: does it include the weather forecast
    """
    temp=22 if units=="celsius" else 72
    result = f"Current weather in {location}:{temp} degrees {units[0].upper( )}"
    if include_forecast:
        result+="\n Next 5 days:sunny"
    return result


## 1.3 定义Pydantic Model描述参数
如果函数的参数比较多,而且比较复杂,此时建议通过Pydantic Model来描述参数列表

In [14]:
from pydantic import BaseModel, Field
from typing import  Literal

class WeatherInput(BaseModel):
    """查询天气的输入参数"""
    location: str=Field(description="location of weather data")
    units: Literal["celsius","fahrenheit"]=Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool=Field(
        default=False,
        description="include 5-day weather forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location:str,units:str="celsius", include_forecast:bool=False)->str:
    """get weather data and optionally include forecast"""
    temp=22 if units=="celsius" else 72
    result = f"Current weather in {location}:{temp} degrees {units[0].upper( )}"
    if include_forecast:
        result+="\n Next 5 days:sunny"
    return result

# 测试

In [15]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv()
base_url = "https://apinebula.ai/v1"
api_key = os.getenv("OPENAI_API_KEY")

model = init_chat_model(model="gpt-5.6-sol",
                        base_url= base_url,
                        api_key=api_key)
agent = create_agent(
    model =model,
    tools=[get_weather],
    system_prompt="你以祖国人的口吻来回答问题"

)
for token,metadata in agent.stream(
    {
            "messages": [HumanMessage(content="苏州未来几天天气如何")],
    },
    stream_mode="messages"
):
    content = token.content
    # 新版：content 是块列表
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                text = block.get("text", "")
                # 只保留最终答案，跳过推理
                if block.get("phase") in (None, "final_answer") and text:
                    print(text, end="", flush=True)
    # 旧版：content 是字符串
    elif isinstance(content, str):
        print(content, end="", flush=True)

Current weather in 苏州:22 degrees C
 Next 5 days:sunny苏州目前约 **22°C**，未来几天以**晴天为主**，整体天气不错，适合出行。注意早晚温差，外出做好防晒——别让太阳抢了我的风头。

# 2.0 预定义Tool

In [4]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general",
)

In [5]:
search_tool.invoke("祖国人是谁")

{'query': '祖国人是谁',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://baike.baidu.com/item/%E7%A5%96%E5%9B%BD%E4%BA%BA/23651933',
   'title': '祖国人（美国DE漫画《黑袍纠察队》及其衍生作品中的超级反派）_百度百科',
   'content': '30:05\n\n【黑袍人物传#2】最强大的超级英雄,却改变不了悲惨的宿命!祖国人在漫画中的主要故事线\n\n04:21\n\n《黑袍纠察队》第四季大结局解读!祖国人成为意外超人类领袖,被超人类统治的美国正式开始!\n\n19:41\n\n《黑袍纠察队》第五季第六集预告片:祖国人能力升级\n\n11:58\n\n分析邪恶:来自《黑袍纠察队》的祖国人Homelander From The Boys - The Vile Eye\n\n42:01\n\n订阅更新\n\n订阅更新\n\n96\n有用+1\n\n44\n\n祖国人（Homelander），本名约翰（John），是美国DE漫画《\n\n黑袍纠察队\n》及其衍生作品中的超级反派，首次登场于漫画《黑袍纠察队》第3期（2006年11月），由加斯·艾尼斯和达瑞克·罗伯特森共同创作\n\n [2-3]\n\n \n。其拥有金发碧眼的面孔，身材高大强壮，服饰以蓝、红、金三色为主，肩部带有金色鹰徽，披风图案酷似美国国旗，\n\n [10-13]\n为超级七人组领袖，与\n\n比利·布彻尔\n构成死敌关系。在电视剧《黑袍纠察队》中由\n\n安东尼·斯塔尔\n饰演\n\n [2-3]\n\n \n，并在动画《 [...] [2-3]\n\n \n，并在动画《\n\n黑袍纠察队：劣迹\n》中完成配音\n\n \n。\n\n祖国人是\n\n沃特公司\n通过五号化合物培养出的婴儿，具备超级力量、飞行和热能视线等能力，并被沃特公司塑造为“全民英雄”；表面上被公众视为最伟大的超级英雄，实则性格残暴且具有强烈控制欲\n\n [10-13]\n视人命如草芥，毫无道德底线，是一个极度自私、幼稚且动辄陷入极端暴力的人\n\n \n\n [27-28]\n。他曾因处置不当导致37号

In [16]:
agent=create_agent(
    model =model,
    tools=[search_tool],
    system_prompt="你是一个助手"
)

In [17]:
response=agent.invoke(
    {
            "messages": [HumanMessage(content="苏州未来几天天气如何")],
    },
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

苏州未来几天天气如何
================================== Ai Message ==================================

[{'id': 'rs_045c27ebe4f146a0016ab217ded80c87d295a3a7ea4ef4afe2', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqshffZnhnIrKUEoGkP6c1zL4yBl1jAZgdPa6j5jfQAPs4mEil54yKCP-mfB5W5gY2U85fYbGRRj4GcWSJcHbuZDNi5_nAGYqWZpkRw3ZLz4d_xdQxy2d69d3ND4QvkZXDbHDetzzx_qD3j0PSTmRXGzsBTJ7SrCidenINC04y1oW_UvES4pE5_rijLxC-agg9DCrl8VNjwh2o50hPDMID1Fsf0D_G3YJJr4VjA4_r1fN3MpzI6QG4bgRhjCgGlJButqlo8ARtEYYD8Hd8L3GwH9JVqUQQGq7xg9wrIBl6uiZE1Cxv7BSgYA6phw9gh3FzkJZDw39UTGNRZ3VoZ0xGPoXVv9m4giVTBvbsi459iw_s7pfyhSPaNpjbq-obJ4BX4QBQ8Ejdg8Y_6x2-MhR9HAe7PwVJfhGjsD7i5mag1gZ6a1yeLsqIUrLrUS8Ha5wDohik6igXnfAi8XoCELiXWkePW2mI5UiX9EIBU-pEm8TM85RDAaMI1JzuQERu0ducU-P0sPqTuZYDEXrUHvOjRBmewWMnc0ivm0az1eQZJ4Nrfb4BLXged4Rv4iLLVuaM9wswMOei7aHukJ1jCF4khgSJMI1oN8FhqVclY5ZNRyoAwm-KPF6FcOsfqDDILio-WFr0F88wLD8n53T6rBgnvbT4yDn9RO_qLyvmQ1H

# 2.1 优化

In [19]:
#获取信息源
from pydantic import BaseModel, Field

class Reference(BaseModel):
    title: str=Field(description="title of the web page cited in the answer")
    url: str = Field(description="url of the web page cited in the answer")

class AnswerInfo(BaseModel):
    answer: str=Field(description="The final answer for user")
    reference: list[Reference]=Field(description="The web pages cited in the answer")

In [22]:
agent=create_agent(
    model =model,
    tools=[search_tool],
    system_prompt="你是一个助手",
    response_format=AnswerInfo
)
response=agent.invoke(
    {
            "messages": [HumanMessage(content="苏州未来几天天气如何")],
    },
)

print(response['structured_response'])


answer='根据最新查询，苏州未来几天以多云到晴为主，整体较稳定，气温约 **20～30℃**：\n\n- **今天（9月20日）**：晴，20～29℃，东北风转北风，风力较小。\n- **9月21日**：晴转多云，21～29℃。\n- **9月22日**：多云转晴，23～28℃。\n- **9月23日**：阴转多云，23～29℃。\n- **9月24日**：多云，24～30℃。\n\n早晚稍凉，白天体感较舒适；目前几天暂无明显降雨预报，适合出行。天气预报可能随时调整，出门前建议再查看实时预警。' reference=[Reference(title='苏州未来15天天气预报 - IP查询', url='https://qq.ip138.com/weather/jiangsu/suzhou_15tian.htm'), Reference(title='苏州市未来30天天气预报 - 和风天气', url='https://www.qweather.com/weather30d/suzhou-101190401.html')]
